In [ ]:
!pip install google-cloud-bigquery google-cloud-bigquery-storage db-dtypes pandas

In [ ]:
# Import libraries
from google.cloud import bigquery
import pandas as pd
from datetime import datetime

In [ ]:
# Initialize BigQuery client
client = bigquery.Client()

In [ ]:
# Configuration - Update these with your project details
PROJECT_ID = "qwiklabs-gcp-04-af615cbd429c"
DATASET_ID = "weather_analysis"
TABLE_ID = "weather_data"
MODEL_ID = "gemini_model"

In [ ]:
# Set the project
client.project = PROJECT_ID

In [ ]:
print(f"Project ID: {PROJECT_ID}")
print(f"Dataset: {DATASET_ID}")
print(f"Table: {TABLE_ID}")

Project ID: qwiklabs-gcp-04-af615cbd429c
Dataset: weather_analysis
Table: weather_data


In [ ]:
# Load CSV file from Google Cloud Storage into BigQuery
table_ref = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}"

# Configure the load job
job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.CSV,
    skip_leading_rows=1,
    autodetect=True,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE
)

# GCS URI
uri = "gs://labs.roitraining.com/data-to-ai-workshop/weather_data.csv"

# Load the data
load_job = client.load_table_from_uri(
    uri,
    table_ref,
    job_config=job_config
)

load_job.result()

print(f"Data loaded successfully into {table_ref}")
print(f"Total rows loaded: {load_job.output_rows}")

Data loaded successfully into qwiklabs-gcp-04-af615cbd429c.weather_analysis.weather_data
Total rows loaded: 300


In [ ]:
# explore the data
explore_query = f"""
SELECT *
FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}`
LIMIT 10
"""

# Execute query and display results
df = client.query(explore_query).to_dataframe()
print("Sample data from weather_data table:")
print(df)
print("\n")
print(f"Shape: {df.shape}")
print("\n")
print("Column names and types:")
print(df.dtypes)
print("\n")
print("Statistical summary:")
print(df.describe())

Sample data from weather_data table:
         date     city state  temperature_f  wind_speed_mph  precipitation_in  \
0  2025-02-21  Atlanta    GA           55.7             5.0              0.12   
1  2025-02-26  Atlanta    GA           75.2            10.4              0.03   
2  2025-03-01  Atlanta    GA           51.7             4.7              0.08   
3  2025-03-05  Atlanta    GA           74.4             5.1              0.02   
4  2025-03-10  Atlanta    GA           59.5             9.6              0.09   
5  2025-03-14  Atlanta    GA           71.7             7.2              0.18   
6  2025-02-19   Boston    MA           61.7             3.9              0.11   
7  2025-03-09   Boston    MA           76.7             4.3              0.09   
8  2025-03-13   Boston    MA           71.9             9.8              0.16   
9  2025-03-19   Boston    MA           60.7             6.4              0.04   

   barometric_pressure_inHg  humidity_percent weather_condition  
0    

In [ ]:
# Get detailed schema information
table = client.get_table(table_ref)

print(f"Table schema for {table_ref}:")
for field in table.schema:
    print(f"  - {field.name}: {field.field_type}")

Table schema for qwiklabs-gcp-04-af615cbd429c.weather_analysis.weather_data:
  - date: DATE
  - city: STRING
  - state: STRING
  - temperature_f: FLOAT
  - wind_speed_mph: FLOAT
  - precipitation_in: FLOAT
  - barometric_pressure_inHg: FLOAT
  - humidity_percent: FLOAT
  - weather_condition: STRING


In [ ]:
import subprocess
import json

PROJECT_ID = "qwiklabs-gcp-04-af615cbd429c"

# Step 1: Create connection
print("Step 1: Creating BigQuery connection...")
result = subprocess.run(
    [
        'bq', 'mk', '--connection',
        '--location=us',
        f'--project_id={PROJECT_ID}',
        '--connection_type=CLOUD_RESOURCE',
        'gemini_connection'
    ],
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print("✓ Connection created successfully")
elif "already exists" in result.stderr:
    print("✓ Connection already exists")
else:
    print(f"Error creating connection: {result.stderr}")

# Step 2: Get service account
print("\nStep 2: Getting service account...")
result = subprocess.run(
    ['bq', 'show', '--connection', '--format=json',
     f'{PROJECT_ID}.us.gemini_connection'],
    capture_output=True,
    text=True
)

if result.returncode == 0:
    try:
        connection_info = json.loads(result.stdout)
        service_account = connection_info.get('cloudResource', {}).get('serviceAccountId')

        if service_account:
            print(f"✓ Service Account: {service_account}")

            # Step 3: Grant permissions
            print("\nStep 3: Granting Vertex AI User role...")
            result = subprocess.run(
                [
                    'gcloud', 'projects', 'add-iam-policy-binding',
                    PROJECT_ID,
                    f'--member=serviceAccount:{service_account}',
                    '--role=roles/aiplatform.user',
                    '--condition=None'
                ],
                capture_output=True,
                text=True
            )

            if result.returncode == 0:
                print("✓ Permissions granted successfully!")
                print("\n" + "="*60)
                print("SETUP COMPLETE! Now you can create your Gemini model.")
                print("="*60)
            else:
                print(f"Warning: {result.stderr}")
                print("\nYou may need to grant permissions manually via the Console.")
        else:
            print("Could not find service account in connection info")
    except json.JSONDecodeError as e:
        print(f"Error parsing connection info: {e}")
else:
    print(f"Error getting service account: {result.stderr}")


Step 1: Creating BigQuery connection...
Error creating connection: 

Step 2: Getting service account...
✓ Service Account: bqcx-413993506028-3vb3@gcp-sa-bigquery-condel.iam.gserviceaccount.com

Step 3: Granting Vertex AI User role...
✓ Permissions granted successfully!

SETUP COMPLETE! Now you can create your Gemini model.


In [ ]:
from google.cloud import bigquery

client = bigquery.Client(project='qwiklabs-gcp-04-af615cbd429c')

query = """
CREATE OR REPLACE MODEL `qwiklabs-gcp-04-af615cbd429c.weather_analysis.gemini_model`
REMOTE WITH CONNECTION `qwiklabs-gcp-04-af615cbd429c.us.gemini_connection`
OPTIONS (
  ENDPOINT = 'gemini-2.5-flash'
)
"""

print("Creating Gemini model...")
job = client.query(query)
job.result()  # Wait for completion

print("Model created successfully!")

Creating Gemini model...
Model created successfully!


In [ ]:
# Create a table with weather reports generated by Gemini
generate_report_query = f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}_with_reports` AS
SELECT
  *
FROM
  ML.GENERATE_TEXT(
    MODEL `{PROJECT_ID}.{DATASET_ID}.{MODEL_ID}`,
    (
      SELECT
        *,
        CONCAT(
          'Generate a weather report for ', city, ', ', state, ' on ', CAST(date AS STRING), '. ',
          'Temperature: ', CAST(temperature_f AS STRING), '°F, ',
          'Condition: ', weather_condition, ', ',
          'Wind Speed: ', CAST(wind_speed_mph AS STRING), ' mph, ',
          'Humidity: ', CAST(humidity_percent AS STRING), '%, ',
          'Precipitation: ', CAST(precipitation_in AS STRING), ' inches, ',
          'Barometric Pressure: ', CAST(barometric_pressure_inHg AS STRING), ' inHg. ',
          'Provide a brief weather report with any warnings if conditions are severe or unusual.'
        ) AS prompt
      FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}`
    ),
    STRUCT(
      0.2 AS temperature,
      512 AS max_output_tokens,
      0.8 AS top_p,
      40 AS top_k
    )
  )
"""

print("Generating weather reports...")
try:
    query_job = client.query(generate_report_query)
    query_job.result()
    print("Weather reports generated successfully!")
except Exception as e:
    print(f"Error: {e}")



Generating weather reports...
Weather reports generated successfully!


In [ ]:
# Query the results
results_query = f"""
SELECT
  date,
  city,
  state,
  temperature_f,
  weather_condition,
  wind_speed_mph,
  humidity_percent,
  precipitation_in,
  barometric_pressure_inHg,
  ml_generate_text_result AS weather_report
FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}_with_reports`
LIMIT 10
"""

# Display results
df_results = client.query(results_query).to_dataframe()
print("Weather Data with Generated Reports:")
print(df_results)

# Display full reports with correct column names
for idx, row in df_results.iterrows():
    print(f"\n{'='*80}")
    print(f"Record {idx + 1}:")
    print(f"Date: {row['date']}")
    print(f"Location: {row['city']}, {row['state']}")
    print(f"Temperature: {row['temperature_f']}°F")
    print(f"Condition: {row['weather_condition']}")
    print(f"Wind Speed: {row['wind_speed_mph']} mph")
    print(f"Humidity: {row['humidity_percent']}%")
    print(f"Precipitation: {row['precipitation_in']} inches")
    print(f"Barometric Pressure: {row['barometric_pressure_inHg']} inHg")
    print(f"\nGenerated Report:")
    print(row['weather_report'])


Weather Data with Generated Reports:
         date     city state  temperature_f weather_condition  wind_speed_mph  \
0  2025-03-01   Boston    MA           75.6             Sunny             0.0   
1  2025-03-12   Boston    MA           23.5             Snowy             9.0   
2  2025-03-18  Seattle    WA           37.2             Snowy             6.0   
3  2025-02-28  Atlanta    GA           70.0             Sunny             3.0   
4  2025-03-18  Atlanta    GA           91.0             Sunny             4.0   
5  2025-03-14   Boston    MA           84.6             Sunny             2.7   
6  2025-03-06   Boston    MA           33.0             Snowy            16.0   
7  2025-03-02  Chicago    IL           68.4             Sunny             3.0   
8  2025-03-12    Miami    FL           85.2             Sunny             3.0   
9  2025-03-08    Miami    FL           91.7             Sunny             5.5   

   humidity_percent  precipitation_in  barometric_pressure_inHg  \
0   

In [ ]:
# Export results to CSV for review
output_file = "weather_reports.csv"
df_results.to_csv(output_file, index=True)
print(f"Results exported to {output_file}")


Results exported to weather_reports.csv
